# 03 — Agente LLM con RAG y herramientas

Adaptado de `lecture10/01_rag_nvidia_nim.ipynb` (RAG) y `Lecture11/part01_from_prompt_to_action.ipynb` .

| Componente | Implementación |
|---|---|
| LLM | NVIDIA NIM `nvidia/nemotron-3.5-super-vl-120b-a12b` (endpoint compatible con OpenAI) |
| Embeddings | NVIDIA NIM `nvidia/nemotron-3-embed-1b`, normalizados |
| Vector store | FAISS `IndexFlatIP` (producto interno = coseno), umbral 0.25 |
| Contexto | 4 resoluciones CREG → chunking semántico (≤ 900 caracteres) |


**Herramientas**

| Herramienta | Qué hace | Conecta con |
|---|---|---|
| `consultar_precios(fecha_inicio, fecha_fin)` | estadísticas reales del precio de bolsa | datos XM |
| `simular(politica, fecha_inicio, fecha_fin, capacidad_kwh, potencia_kw, eficiencia)` | ejecuta el MDP y compara políticas | **Parte 1** (Q-learning / iteración de valor) |
| `buscar_documentos(consulta)` | RAG sobre resoluciones CREG | corpus regulatorio |

**Configuración:** la clave se lee de `.env` (`NVIDIA_API_KEY`);

In [1]:
import sys, json
sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from config import MODEL, EMBEDDING_MODEL
print("LLM:", MODEL, "| Embeddings:", EMBEDDING_MODEL)

LLM: nvidia/nemotron-3.5-super-vl-120b-a12b | Embeddings: nvidia/nemotron-3-embed-1b


## 1. Índice vectorial

`get_vector_store()` carga el índice de `data/index/` y, si no existe, lo construye
(descarga los documentos, hace el chunking, calcula los embeddings y arma el índice FAISS).

In [3]:
from rag import get_vector_store
store = get_vector_store()
print("Chunks indexados:", store.index.ntotal)
pd.Series([c["title"] for c in store.chunks]).value_counts()

Chunks indexados: 960


Resolución CREG 101 113 de 2026 (SAEB en el SIN)                 368
Resolución CREG 024 de 1995 (reglas de la bolsa de energía)      268
Resolución CREG 174 de 2021 (autogeneración a pequeña escala)    236
Resolución CREG 098 de 2019 (SAEB, derogada)                      88
Name: count, dtype: int64

## 2. Recuperación: relevancia y umbral

Una pregunta fuera del dominio (tarifas de EPM) no recupera fragmentos por encima del umbral.
Así el agente puede reconocer que no tiene contexto.

In [4]:
for q in ["¿Qué es un SAEB?", "liquidación de la energía de SAEB activos de red", "tarifa de EPM estrato 3"]:
    hits = store.search(q, k=3)
    print(f"\n❓ {q}  →  {len(hits)} fragmentos")
    for h in hits:
        print(f"   [{h['score']:.3f}] {h['title']}: {h['content'][:100]!r}")


❓ ¿Qué es un SAEB?  →  3 fragmentos
   [0.480] Resolución CREG 098 de 2019 (SAEB, derogada): 'ARTÍCULO 4o. IDENTIFICACIÓN DE LA NECESIDAD DE UN SAEB. <Resolución derogada por el artículo 108 de '
   [0.448] Resolución CREG 101 113 de 2026 (SAEB en el SIN): 'Sistema de almacenamiento de energía eléctrica, SAE: es un sistema de almacenamiento de energía de c'
   [0.431] Resolución CREG 101 113 de 2026 (SAEB en el SIN): 'Durante la operación real y ante contingencias, el SAEB cumplirá con la respuesta operativa conforme'



❓ liquidación de la energía de SAEB activos de red  →  3 fragmentos
   [0.630] Resolución CREG 101 113 de 2026 (SAEB en el SIN): 'Para los SAEB como activos de red instalados en el SDL, el ASIC hará la liquidación horaria de la en'
   [0.628] Resolución CREG 101 113 de 2026 (SAEB en el SIN): 'Para los SAEB como activos de red instalados en el STN o el STR, independientemente de su mecanismo '
   [0.584] Resolución CREG 101 113 de 2026 (SAEB en el SIN): 'b) En el caso de los SAEB como activos de red, la energía requerida para arranque autónomo deberá se'



❓ tarifa de EPM estrato 3  →  0 fragmentos


## 3. Herramientas ejecutadas directamente (sin LLM)

In [5]:
from agente import consultar_precios, simular, buscar_documentos, _execute_tool
print(consultar_precios("2026-03-01", "2026-03-31")[:400])

{"fecha_inicio": "2026-03-01", "fecha_fin": "2026-03-31", "dias": 31, "precio_promedio_cop_kwh": 220.2, "precio_min_cop_kwh": 101.7, "precio_max_cop_kwh": 1135.36, "hora_mas_barata_promedio": 9, "hora_mas_cara_promedio": 19, "diferencial_promedio_diario_cop_kwh": 235.99, "perfil_horario_promedio": {"0": 198.6, "1": 198.0, "2": 198.4, "3": 195.6, "4": 200.3, "5": 208.6, "6": 205.5, "7": 197.7, "8":


In [6]:
print(json.dumps(json.loads(simular("q_learning", "2026-01-01", "2026-10-31"))["comparacion"], ensure_ascii=False, indent=1))

{
 "Q-learning": {
  "ahorro_total_cop": 484607,
  "pct_del_oraculo": 74.6,
  "ciclos_equivalentes": 346.0
 },
 "Iteración de valor": {
  "ahorro_total_cop": 492820,
  "pct_del_oraculo": 75.8,
  "ciclos_equivalentes": 301.5
 },
 "Horario fijo (ingenua)": {
  "ahorro_total_cop": 469599,
  "pct_del_oraculo": 72.3,
  "ciclos_equivalentes": 304.0
 },
 "Aleatoria": {
  "ahorro_total_cop": -981480,
  "pct_del_oraculo": -151.1,
  "ciclos_equivalentes": 484.2
 },
 "Oráculo (info. perfecta)": {
  "ahorro_total_cop": 649756,
  "pct_del_oraculo": 100.0,
  "ciclos_equivalentes": 277.5
 }
}


### Manejo de errores: el error vuelve al modelo como observación

In [7]:
print(_execute_tool("simular", '{"politica": "magia", "fecha_inicio": "2026-01-01", "fecha_fin": "2026-02-01"}')[1])
print(_execute_tool("consultar_precios", '{"fecha_inicio": "2030-01-01", "fecha_fin": "2030-01-31"}')[1])
print(_execute_tool("no_existe", '{}')[1])

Error en simular: Política desconocida 'magia'. Opciones: ['q_learning', 'iteracion_valor', 'horario_fijo', 'aleatoria', 'oraculo']
Error en consultar_precios: No hay datos entre 2030-01-01 y 2030-01-31. Rango disponible: 2024-01-01 a 2026-10-31
Error: herramienta 'no_existe' no encontrada.


## 4. El agente en acción

Se muestra la herramienta llamada, sus argumentos, el resultado y la respuesta final.

In [8]:
from agente import run_agent

def show(question):
    out = run_agent(question)
    print("👤", question)
    for t in out["trace"]:
        print(f"🔧 {t['tool']}({json.dumps(t['args'], ensure_ascii=False)})")
        print("   →", t["result"][:250], "...")
    print(f"\n🤖 ({out['iterations']} iteraciones)\n{out['answer']}")

show("¿Cuánto ahorra Q-learning frente al horario fijo con una batería de 20 kWh y 5 kW entre 2026-01-01 y 2026-10-31?")

👤 ¿Cuánto ahorra Q-learning frente al horario fijo con una batería de 20 kWh y 5 kW entre 2026-01-01 y 2026-10-31?
🔧 simular({"politica": "q_learning", "fecha_inicio": "2026-01-01", "fecha_fin": "2026-10-31", "capacidad_kwh": 20, "potencia_kw": 5, "eficiencia": 0.9})
   → {"politica": "Q-learning", "parametros": {"capacidad_kwh": 20, "potencia_kw": 5, "eficiencia": 0.9}, "periodo": {"inicio": "2026-01-01", "fin": "2026-10-31", "dias": 304}, "advertencia": "Periodo de prueba (no visto en entrenamiento).", "comparacion" ...
🔧 simular({"politica": "horario_fijo", "fecha_inicio": "2026-01-01", "fecha_fin": "2026-10-31", "capacidad_kwh": 20, "potencia_kw": 5, "eficiencia": 0.9})
   → {"politica": "Horario fijo (ingenua)", "parametros": {"capacidad_kwh": 20, "potencia_kw": 5, "eficiencia": 0.9}, "periodo": {"inicio": "2026-01-01", "fin": "2026-10-31", "dias": 304}, "advertencia": "Periodo de prueba (no visto en entrenamiento).", " ...

🤖 (2 iteraciones)
Con una batería de 20 kWh y 5 kW entre

In [9]:
show("¿Qué dice la CREG sobre la eficiencia mínima de los SAEB?")

👤 ¿Qué dice la CREG sobre la eficiencia mínima de los SAEB?
🔧 buscar_documentos({"consulta": "eficiencia mínima SAEB sistemas almacenamiento energía baterías CREG resolución 101 113 2026 098 2019"})
   → [{"fuente": "Resolución CREG 101 113 de 2026 (SAEB en el SIN)", "url": "https://gestornormativo.creg.gov.co/gestor/entorno/docs/resolucion_creg_101-113_2026.htm", "score": 0.664, "texto": "RESOLUCIÓN 101 113 DE 2026\n\n(junio 19)\n\n<Publicado en la  ...
🔧 buscar_documentos({"consulta": "porcentaje eficiencia mínima SAEB ciclo completo resolución CREG 098 2019 101 113 2026 valor"})
   → [{"fuente": "Resolución CREG 098 de 2019 (SAEB, derogada)", "url": "https://gestornormativo.creg.gov.co/gestor/entorno/docs/resolucion_creg_0098_2019.htm", "score": 0.62, "texto": "ARTÍCULO 3o. DEFINICIONES. <Resolución derogada por el artículo 108 d ...

🤖 (3 iteraciones)
Según los fragmentos de las resoluciones de la CREG consultados:

**Resolución CREG 101 113 de 2026 (vigente):**
- El **Artículo 6°*

### Encadenamiento de herramientas (RAG + simulación)

In [10]:
show("¿Qué es el precio de bolsa según la regulación y cuál fue su promedio en agosto de 2026?")

👤 ¿Qué es el precio de bolsa según la regulación y cuál fue su promedio en agosto de 2026?
🔧 buscar_documentos({"consulta": "precio de bolsa definición regulación CREG"})
   → [{"fuente": "Resolución CREG 024 de 1995 (reglas de la bolsa de energía)", "url": "https://gestornormativo.creg.gov.co/gestor/entorno/docs/resolucion_creg_0024_1995.htm", "score": 0.462, "texto": "<Aparte modificado por el artículo 28 de la Resolució ...
🔧 consultar_precios({"fecha_inicio": "2026-08-01", "fecha_fin": "2026-08-31"})
   → {"fecha_inicio": "2026-08-01", "fecha_fin": "2026-08-31", "dias": 31, "precio_promedio_cop_kwh": 943.55, "precio_min_cop_kwh": 132.1, "precio_max_cop_kwh": 1069.94, "hora_mas_barata_promedio": 8, "hora_mas_cara_promedio": 19, "diferencial_promedio_di ...

🤖 (2 iteraciones)
Según la regulación, el precio de bolsa es el precio horario determinado en la Bolsa de Energía, igual al precio de oferta de la planta con máximo precio de oferta en la hora respectiva, más el Valor Adicional d

## 5. Límite de iteraciones

Con `T_MAX` iteraciones el loop se detiene y devuelve
`[Agente detenido: se alcanzó el límite de T_MAX iteraciones]`. Así se evita un ciclo infinito si el
modelo sigue pidiendo herramientas. Aquí se fuerza con `T_MAX = 1` y una pregunta que necesita herramientas.

In [11]:
import agente
agente.T_MAX = 1
print(run_agent("¿Cuál fue el precio promedio en marzo de 2026?")["answer"])
agente.T_MAX = 6

[Agente detenido: se alcanzó el límite de 1 iteraciones]
